# 00 Descarga y preparación

Este notebook prepara el dataset **APS Failure at Scania Trucks**. El objetivo del proyecto es detectar fallos relacionados con el sistema APS de camiones Scania a partir de variables tabulares de sensores. La clase positiva representa un fallo APS y es la clase importante desde el punto de vista operativo.

En mantenimiento predictivo, dejar escapar un fallo APS puede ser mucho más grave que revisar un camión que finalmente no tiene ese fallo. Por eso desde el principio se guarda una definición clara de la clase positiva y del coste de los errores. Este notebook solo prepara los datos. **No entrena modelos y no usa test para tomar decisiones**.
El sistema APS está relacionado con el circuito de aire a presión del camión. Un fallo en este sistema puede afectar al funcionamiento del vehículo y puede generar costes de mantenimiento importantes. Por eso el problema no se plantea como una clasificación abstracta, sino como una decisión de apoyo a mantenimiento.

La clase positiva es poco frecuente, pero es la clase que más interesa detectar. Esta característica hace que el proyecto tenga una dificultad real. No basta con construir un modelo que acierte muchos casos normales. El modelo debe encontrar señales de fallo en una minoría de ejemplos.

## Librerías y carpetas

Esta primera celda carga las librerías básicas del proyecto y las funciones auxiliares de **project_utils.py**. También crea la estructura de carpetas donde se guardan datos procesados, métricas, figuras y modelos.

La celda no analiza resultados ni ajusta modelos. Su papel es asegurar que todos los notebooks posteriores trabajen con la misma organización de archivos. Esto mejora la reproducibilidad y evita rutas sueltas difíciles de defender.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from project_utils import *

RANDOM_STATE = 42
crear_carpetas(".")
sns.set_theme(style="whitegrid")

La configuración inicial deja preparado el entorno de trabajo. Todavía no se han cargado datos y no se ha usado ninguna partición. Por tanto, no hay ninguna decisión experimental tomada en esta parte.

## Carga del dataset

En esta sección se cargan los archivos oficiales del dataset APS. Si los datos ya existen en **data/raw**, se leen desde local. Si no existen, se descargan desde UCI y se guardan para no depender de una descarga en cada ejecución.

El dataset proporciona una partición oficial de train y otra de test. Mantener esa separación es importante porque el criterio del profesor exige que test quede reservado para la evaluación final. A partir de aquí, train servirá para explorar, ajustar hiperparámetros y seleccionar modelos. **Test no se usará para elegir nada**.
Usar los splits oficiales facilita que el experimento sea reproducible y comparable. También evita una decisión adicional sobre cómo separar los datos. En este proyecto esa separación queda fijada desde el inicio y no se revisa después de ver resultados.

Esta elección conecta directamente con el criterio del profesor. Primero se prepara train y test. Después todo el ajuste se hará con train. Test se reserva como una muestra independiente para medir el rendimiento final.

In [ ]:
train_raw, test_raw = cargar_o_descargar_dataset("data/raw")
print("Train raw:", train_raw.shape)
print("Test raw:", None if test_raw is None else test_raw.shape)
train_raw.head()

Train raw: (60000, 171)
Test raw: (16000, 171)


,class,aa_000,ab_000,ac_000,ad_000,ae_000,af_000,ag_000,ag_001,ag_002,...,ee_002,ee_003,ee_004,ee_005,ee_006,ee_007,ee_008,ee_009,ef_000,eg_000
0,neg,76698,NaN,2.130706e+09,280.0,0.0,0.0,0.0,0.0,0.0,...,1240520.0,493384.0,721044.0,469792.0,339156.0,157956.0,73224.0,0.0,0.0,0.0
1,neg,33058,NaN,0.000000e+00,NaN,0.0,0.0,0.0,0.0,0.0,...,421400.0,178064.0,293306.0,245416.0,133654.0,81140.0,97576.0,1500.0,0.0,0.0
2,neg,41040,NaN,2.280000e+02,100.0,0.0,0.0,0.0,0.0,0.0,...,277378.0,159812.0,423992.0,409564.0,320746.0,158022.0,95128.0,514.0,0.0,0.0
3,neg,12,0.0,7.000000e+01,66.0,0.0,10.0,0.0,0.0,0.0,...,240.0,46.0,58.0,44.0,10.0,0.0,0.0,0.0,4.0,32.0
4,neg,60874,NaN,1.368000e+03,458.0,0.0,0.0,0.0,0.0,0.0,...,622012.0,229790.0,405298.0,347188.0,286954.0,311560.0,433954.0,1218.0,0.0,0.0


La carga confirma que se trabaja con un conjunto grande y tabular. En la ejecución actual aparecen **60000 filas de train** y **16000 filas de test**. También aparecen **170 variables predictoras**, lo que da suficiente riqueza para comparar modelos lineales, árboles y boosting.

El tamaño del dataset permite una validación cruzada razonable en train. La existencia de test separado permite hacer una evaluación final limpia al terminar la selección de modelos.
El número de variables también justifica que el proyecto compare varias familias de modelos. Con tantas señales, es razonable que haya relaciones lineales, interacciones y patrones no lineales. Por eso más adelante se prueban modelos lineales, árboles y boosting.

El tamaño de test es suficiente para una evaluación final informativa. Aun así, test no se interpreta como una herramienta para elegir. Su función es medir una vez el comportamiento de modelos ya fijados.

## Limpieza inicial

Esta sección normaliza los nombres de las columnas y convierte los valores marcados como **na** en valores faltantes reales. En el archivo original, **na** no es una categoría ni un valor numérico, sino la forma en la que el dataset indica que falta una medición.

También convierte la variable objetivo a formato numérico. En el archivo original, la etiqueta **neg** indica que el fallo no está relacionado con el sistema APS, mientras que la etiqueta **pos** indica que sí hay un fallo APS. Para trabajar de forma más cómoda con los modelos, **neg** se transforma en 0 y **pos** se transforma en 1. Así, la clase 1 queda como la clase positiva y representa el caso más importante del proyecto.

Este paso es necesario para que todos los modelos trabajen con una variable objetivo clara. La limpieza se aplica de forma homogénea a train y test, pero no aprende parámetros estadísticos de test. Por eso no introduce fuga de información.

In [ ]:
train_raw = convertir_faltantes(limpiar_nombres_columnas(train_raw))
if test_raw is not None:
    test_raw = convertir_faltantes(limpiar_nombres_columnas(test_raw))

if train_raw["class"].dtype == object:
    train_raw["class"] = train_raw["class"].astype(str).str.lower().map({"neg": 0, "pos": 1})
if test_raw is not None and test_raw["class"].dtype == object:
    test_raw["class"] = test_raw["class"].astype(str).str.lower().map({"neg": 0, "pos": 1})

print(train_raw["class"].value_counts())
print("Duplicados:", train_raw.duplicated().sum())
train_raw.isna().mean().sort_values(ascending=False).head(15)

class
0    59000
1     1000
Name: count, dtype: int64
Duplicados: 0


br_000    0.821067
bq_000    0.812033
bp_000    0.795667
bo_000    0.772217
ab_000    0.772150
cr_000    0.772150
bn_000    0.733483
bm_000    0.659150
bl_000    0.454617
bk_000    0.383900
ch_000    0.247683
co_000    0.247683
cg_000    0.247683
cf_000    0.247683
ad_000    0.247683
dtype: float64

La distribución de clases muestra un desbalanceo muy fuerte. En train hay muchos más casos negativos que positivos. Esto es coherente con el problema real, porque los fallos APS son eventos poco frecuentes en comparación con el resto de casos.

Esta observación anticipa una limitación importante. Un modelo puede obtener una accuracy muy alta prediciendo casi siempre la clase negativa. Por eso el proyecto no puede evaluarse solo con accuracy. También deben usarse métricas centradas en la clase positiva, como recall, F1 y PR-AUC.

La proporción de positivos obliga a pensar en el impacto de los errores. Si el modelo falla muchos positivos, puede parecer correcto en una métrica global y ser poco útil para el objetivo real. En este problema, un falso negativo implica no detectar un fallo APS real. Por eso se incorpora una evaluación con coste asimétrico.

Este desbalanceo se tratará en los notebooks posteriores mediante varias decisiones. Primero, se usará un baseline para mostrar el comportamiento de un modelo trivial. Después, los modelos se evaluarán con métricas adecuadas para clases desbalanceadas. También se compararán estrategias con pesos de clase y se analizará el coste de falsos positivos y falsos negativos.

El desbalanceo también condiciona el baseline. Un baseline simple es necesario porque permite mostrar que un resultado aparentemente alto en accuracy puede esconder un rendimiento muy bajo en la clase APS.

## Guardado de train y test

Esta sección guarda los datos procesados en **data/processed**. Se conservan los splits oficiales cuando están disponibles. En este caso, el conjunto de train tiene 60000 instancias y el conjunto de test tiene 16000 instancias. Si no existieran, la separación se haría una sola vez al principio y después se respetaría durante todo el proyecto.

Guardar **train.csv** y **test.csv** permite que los notebooks siguientes sean reproducibles. También evita volver a descargar o limpiar los datos en cada notebook. **El test queda reservado desde este punto para la evaluación final**.

In [ ]:
if test_raw is None:
    train_df, test_df = crear_train_test(train_raw, test_size=0.2, random_state=RANDOM_STATE)
else:
    train_df = train_raw.copy()
    test_df = test_raw.copy()

guardar_csv(train_df, "data/processed/train.csv")
guardar_csv(test_df, "data/processed/test.csv")

info = {
    "dataset": "APS Failure at Scania Trucks",
    "uci_id": 421,
    "n_train": int(train_df.shape[0]),
    "n_test": int(test_df.shape[0]),
    "n_features": int(train_df.shape[1] - 1),
    "fp_cost": 10,
    "fn_cost": 500,
}
guardar_json(info, "artifacts/metrics/dataset_info.json")
info

{'dataset': 'APS Failure at Scania Trucks',
 'uci_id': 421,
 'n_train': 60000,
 'n_test': 16000,
 'n_features': 170,
 'fp_cost': 10,
 'fn_cost': 500}

Los archivos procesados están listos para continuar. También se guarda información básica del dataset y la función de coste usada en el proyecto. El falso positivo cuesta 10 y el falso negativo cuesta 500.

Esta diferencia de coste refleja el contexto de mantenimiento. Revisar un camión sin fallo APS tiene un coste operativo, pero no detectar un fallo APS puede provocar una avería más grave. Esta idea será central en la evaluación posterior.

El guardado de archivos intermedios también separa responsabilidades entre notebooks. Este notebook deja datos limpios y particiones estables. Los siguientes notebooks pueden centrarse en EDA, modelado y auditoría sin repetir pasos de preparación.

Esta organización reduce errores y ayuda a defender la reproducibilidad. Si se ejecuta el proyecto desde cero, todos los notebooks usan los mismos ficheros procesados y la misma definición de clase positiva.